In [ ]:
# Example 1: rignore.walk over a tree with a .gitignore that excludes a file.
import os
import tempfile

import rignore

with tempfile.TemporaryDirectory() as tmp:
    os.makedirs(os.path.join(tmp, 'src'))
    for rel in ('src/keep.py', 'src/ignored.log', 'README.md'):
        with open(os.path.join(tmp, rel), 'w') as f:
            f.write('x')
    with open(os.path.join(tmp, '.gitignore'), 'w') as f:
        f.write('*.log\n')

    # require_git=False so .gitignore is honored outside a real git repo.
    paths = list(rignore.walk(tmp, require_git=False))
    names = sorted(os.path.basename(p) for p in paths)
    print('walked:', names)
    assert 'ignored.log' not in names, 'gitignore did not exclude *.log'
    assert 'keep.py' in names
    assert 'README.md' in names


In [ ]:
# Example 2: rignore.Walker iterator interface.
import os
import tempfile

import rignore

with tempfile.TemporaryDirectory() as tmp:
    for rel in ('a.txt', 'b.txt', 'c.bin'):
        with open(os.path.join(tmp, rel), 'w') as f:
            f.write('data')

    walker = rignore.Walker(tmp)
    seen = []
    for entry in walker:
        seen.append(os.path.basename(entry))
    print('Walker yielded:', sorted(seen))
    assert {'a.txt', 'b.txt', 'c.bin'}.issubset(set(seen))


In [ ]:
# Example 3: ignore_hidden flag controls dotfile visibility.
import os
import tempfile

import rignore

with tempfile.TemporaryDirectory() as tmp:
    with open(os.path.join(tmp, 'visible.txt'), 'w') as f:
        f.write('hi')
    with open(os.path.join(tmp, '.hidden'), 'w') as f:
        f.write('hi')

    hidden_off = sorted(os.path.basename(p) for p in rignore.walk(tmp, ignore_hidden=True))
    hidden_on = sorted(os.path.basename(p) for p in rignore.walk(tmp, ignore_hidden=False))
    print('ignore_hidden=True ->', hidden_off)
    print('ignore_hidden=False ->', hidden_on)
    assert '.hidden' not in hidden_off
    assert '.hidden' in hidden_on


In [ ]:
# Example 4: max_depth caps recursion depth.
import os
import tempfile

import rignore

with tempfile.TemporaryDirectory() as tmp:
    deep = os.path.join(tmp, 'a', 'b', 'c')
    os.makedirs(deep)
    with open(os.path.join(tmp, 'top.txt'), 'w') as f:
        f.write('1')
    with open(os.path.join(tmp, 'a', 'mid.txt'), 'w') as f:
        f.write('2')
    with open(os.path.join(deep, 'deep.txt'), 'w') as f:
        f.write('3')

    shallow = [os.path.basename(p) for p in rignore.walk(tmp, max_depth=1)]
    full = [os.path.basename(p) for p in rignore.walk(tmp)]
    print('depth=1:', sorted(shallow))
    print('full:', sorted(full))
    assert 'deep.txt' in full
    assert 'deep.txt' not in shallow


In [ ]:
# Example 5: overrides — gitignore-style patterns (with '!' to exclude).
import os
import tempfile

import rignore

with tempfile.TemporaryDirectory() as tmp:
    for rel in ('keep.py', 'skip.tmp', 'notes.md'):
        with open(os.path.join(tmp, rel), 'w') as f:
            f.write('x')

    # Overrides: '!*.tmp' excludes *.tmp files from the walk.
    names = sorted(
        os.path.basename(p)
        for p in rignore.walk(tmp, overrides=['!*.tmp'])
    )
    print('after overrides=[\"!*.tmp\"]:', names)
    assert 'skip.tmp' not in names
    assert 'keep.py' in names
    assert 'notes.md' in names
    print('rignore smoke test PASSED')
